In [1]:
import torch
from PIL import Image
from pathlib import Path
import json
import os, sys
import time

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(repo_root)
sys.path.append(os.path.join(repo_root, "src"))

from annotation_methods.io_utils import write_coco_output, retrieve_image_batches

from rex_omni import RexOmniVisualize, RexOmniWrapper


In [2]:
# ---- Reusable constants ----
DATA_ROOT = Path("../../Data")

RESULTS_PATH = Path("../../Results/Experiment_1")

DATASETS = ["apples", "tomatoes"]
DATASET_DICT = {
    "apples": ["good apple", "bad apple"],
    "tomatoes": ["tomato"],
}

In [3]:
def initiate_rex_model(backend:str):
    if backend == "transformers":
        # Initialize the wrapper (model loads internally)
        rex_model = RexOmniWrapper(
            model_path="IDEA-Research/Rex-Omni",   # HF repo or local path
            backend="transformers",                # or "vllm" for high-throughput inference
            attn_implementation="sdpa",  # normally uses flash attn but CUDA driver incompatability
            max_tokens=2048,
            temperature=0.0,
            top_p=0.05,
            top_k=1,
            repetition_penalty=1.05,
        )
    if backend == "vllm":
        # Initialize the wrapper (model loads internally)
        rex_model = RexOmniWrapper(
            model_path="IDEA-Research/Rex-Omni",   # HF repo or local path
            backend="vllm",
            max_tokens=2048,
            temperature=0.0,
            top_p=0.05,
            top_k=1,
            repetition_penalty=1.05,
        )        
    return rex_model

In [6]:
def run_rex_inference_to_coco(
    dataset: str,
    rex_backend: str,
    data_root: Path,
    output_path: Path,
    categories: list,
    batch_size: int = 4,
    sample_size: int | None = 12,
    model_name: str = "rex_omni",
):
    """
    Run REX detection inference on a test split and write COCO-format JSON.

    Parameters
    ----------
    dataset : str
        Dataset name, used to look up categories and paths.
    rex_backend : str
        Backend identifier for initiate_rex_model.
    data_root : Path
        Root path of all datasets (parent of `<dataset>/images/test`).
    output_path : Path
        Directory where the COCO JSON and logs will be written.
    categories : list
        List of category names.
    batch_size : int, optional
        Batch size for retrieve_image_batches.
    sample_size : int | None, optional
        Sample size for retrieve_image_batches (pass None for all images).
    model_name : str, optional
        Name to pass to write_coco_output.

    Returns
    -------
    out_file : str or Path
        Path returned by write_coco_output.
    """

    # Resolve paths and categories
    images_folder = data_root / dataset / "images" / "test"
    categories = categories
    output_path = output_path

    images = []
    annotations = []

    img_id = 1
    ann_id = 1
    num_pred_boxes = 0
    total_inf_time = 0.0

    # Load the model
    rex_model = initiate_rex_model(rex_backend)

    # Run inference
    for batch_images, names in retrieve_image_batches(
        images_folder, batch_size=batch_size, sample_size=sample_size
    ):
        start = time.perf_counter()
        results = rex_model.inference(
            images=batch_images, task="detection", categories=categories
        )
        end = time.perf_counter()
        total_inf_time += (end - start)

        # COCO conversion
        for name, res, im in zip(names, results, batch_images):
            # Skip if inference failed
            if not res.get("success", False):
                continue

            # Original image size: (width, height)
            w, h = res["image_size"]

            # Register image entry
            current_img_id = img_id
            images.append(
                {
                    "id": current_img_id,
                    "file_name": f"images/test/{name}",
                }
            )
            img_id += 1

            # Extract predictions for this image
            preds = res.get("extracted_predictions", {})
            for label, objs in preds.items():
                # Map category name -> category_id (as used in write_coco_output)
                if label not in categories:
                    # If the label is unknown for some reason, skip it
                    continue
                category_id = categories.index(label)

                for obj in objs:
                    if obj.get("type") != "box":
                        continue

                    x0, y0, x1, y1 = obj["coords"]

                    # Clamp coordinates to image bounds
                    x0 = max(0.0, min(float(x0), float(w)))
                    x1 = max(0.0, min(float(x1), float(w)))
                    y0 = max(0.0, min(float(y0), float(h)))
                    y1 = max(0.0, min(float(y1), float(h)))

                    # Convert [x0, y0, x1, y1] -> [x, y, width, height]
                    bbox_w = max(0.0, x1 - x0)
                    bbox_h = max(0.0, y1 - y0)
                    area = bbox_w * bbox_h

                    # Skip degenerate boxes
                    if bbox_w <= 0 or bbox_h <= 0:
                        continue

                    annotations.append(
                        {
                            "id": ann_id,
                            "image_id": current_img_id,
                            "category_id": category_id,
                            "bbox": [x0, y0, bbox_w, bbox_h],
                            "area": area,
                            "iscrowd": 0,
                        }
                    )
                    ann_id += 1
                    num_pred_boxes += 1

    # ---------------------------------------------------------------------
    # Write COCO JSON and return path
    # ---------------------------------------------------------------------
    num_images = len(images)

    out_file = write_coco_output(
        images_folder=str(images_folder),
        model_name=model_name,
        categories_list=categories,
        images=images,
        annotations=annotations,
        num_images=num_images,
        output_path=output_path,
        num_initial_bbox=0,  # zero-shot has no initial boxes
        num_pred_boxes=num_pred_boxes,
        total_inf_time=total_inf_time,
        total_train_time=0.0,  # zero-shot requires no training time
    )

    print(f"Wrote COCO-format JSON to {out_file}")
    return out_file


In [7]:
dataset = "apples"
rex_backend = "transformers"
rex_backend = "vllm"
categories = DATASET_DICT.get(dataset)

run_rex_inference_to_coco(dataset=dataset, rex_backend=rex_backend,data_root=DATA_ROOT,output_path=RESULTS_PATH,categories=categories,batch_size=4,sample_size=12)

Initializing vllm backend...


/home/warredv/miniconda3/envs/rexomni/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 12-27 17:04:36 [__init__.py:244] Automatically detected platform cuda.


2025-12-27 17:04:42,457	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 12-27 17:05:00 [config.py:823] This model supports multiple tasks: {'reward', 'score', 'embed', 'generate', 'classify'}. Defaulting to 'generate'.
INFO 12-27 17:05:02 [config.py:3268] Downcasting torch.float32 to torch.bfloat16.
INFO 12-27 17:05:09 [config.py:2195] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 12-27 17:05:12 [tokenizer.py:262] Using a slow tokenizer. This might cause a significant slowdown. Consider using a fast tokenizer instead.
INFO 12-27 17:05:12 [core.py:455] Waiting for init message from front-end.
INFO 12-27 17:05:12 [core.py:70] Initializing a V1 LLM engine (v0.9.1) with config: model='IDEA-Research/Rex-Omni', speculative_config=None, tokenizer='IDEA-Research/Rex-Omni', skip_tokenizer_init=False, tokenizer_mode=slow, revision=None, override_neuron_config={}, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, di

...

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Found 31 image files


Processed prompts: 100%|██████████| 4/4 [00:03<00:00,  1.02it/s, est. speed input: 2615.83 toks/s, output: 82.00 toks/s]


Wrote COCO-format JSON to ../../Results/Experiment_1/apples_test_rex_omni_predictions.json


'../../Results/Experiment_1/apples_test_rex_omni_predictions.json'